# CORA — Apertus classifier for the Swiss Federal Archives: demonstration

This notebook runs the pipeline on one record live and then explores the 100-record AI-ready dataset.

Pipeline: **sample → fetch (original fields only) → classify (Apertus pass 1) → audit (Apertus pass 2 + rule detectors) → enrich (gazetteer, Wikidata, story trail) → QC → export**.

Requirements: `pip install -r requirements.txt` and an `.env` with `LLM_API_KEY` (see `.env.example`).

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import pandas as pd
from cora import config, lindas, classify, enrich
from cora.apertus import Apertus

cfg = config.load()
llm = cfg["llm"]
ap = Apertus(llm["base_url"], llm["api_key"], llm["model"], llm["temperature"], llm["max_tokens"], llm["retries"])
print("model:", llm["model"], "| endpoint:", llm["base_url"])

model: apertus-v1-5-70b | endpoint: https://gateway.keera.ch/api/v1


## 1. Fetch one record (original archive-database fields only)
Record 3237439: a First World War glass-plate photograph. LINDAS is only the transport; fields LINDAS adds (ISIL holder, Wikimedia links, access-rule vocabulary) are ignored.

In [2]:
rec = lindas.fetch_records(["3237439"])[0]
print(json.dumps(classify.compact_record(rec), ensure_ascii=False, indent=1))

{
 "title": "Zurücknehmen der Geschütze aus der Stellung",
 "reference_code": "E27#1000/721#14095#409*",
 "scope_and_content": "Artillerie Batterie 38",
 "coverage": "Geografische Abdeckung: Plenjouse, Pkt 706",
 "form": "Fotografie",
 "format": "Negativ; Glasplatte; Silberbromid; 13x18cm",
 "date": {
  "from": "1914-01-01",
  "to": "1918-12-31",
  "expressed": "1914.01.01-1918.12.31"
 },
 "accessions": [
  {
   "number": "1000/721",
   "title": "1000/00721 Bundesverwaltung (Bern) (1600-1960)"
  }
 ],
 "hierarchy_top_down": [
  "Archive: Gesamtbestand des Schweizerischen Bundesarchivs (1798-) [CH-BAR#Unterlagen*]",
  "Main Department: Archiv des Schweizerischen Bundesstaates ab 1848 [E*]",
  "Stock: Landesverteidigung (1848-1950) [E27*]",
  "Serie: Stabsdienste, Dienstzweige, Hilfsdienste. Einsatz der Armee [E27#06]",
  "Serie: Massnahmen bei aussenpolitischen Spannungen, Grenzbesetzungen, Aktivdienste [E27#06.H]",
  "Serie: Aktivdienst 1914-1918 [E27#06.H.3]",
  "Serie: Aktivdienstber

## 2. Pass 1 — classification by Apertus v1.5 70B

In [3]:
raw, meta = ap.chat_json(classify.build_messages(rec, cfg))
result, issues = classify.normalize(raw, rec, cfg)
print(f"{meta['seconds']:.1f}s, schema issues: {issues}")
print(json.dumps({k: result[k] for k in ("resume", "title_en", "document_type", "medium", "source_level",
                                        "source_level_rationale", "creator", "date_created", "places")},
                 ensure_ascii=False, indent=1))

17.3s, schema issues: []
{
 "resume": "This is a World War I-era photograph showing the withdrawal of artillery guns from a position at Plenjouse, Point 706, taken by the Swiss Army's geographic-archival section. It documents military operations during the Swiss Army's mobilization period of 1914-1918.",
 "title_en": "Withdrawal of artillery guns from the position",
 "document_type": "photograph",
 "medium": "still image",
 "source_level": "primary",
 "source_level_rationale": "The photograph was taken by the Swiss Army's geographic-archival section during the events it depicts, serving as direct visual documentation of military operations.",
 "creator": "Swiss Army geographic-archival section",
 "date_created": "1914-01-01/1918-12-31",
 "places": [
  {
   "name": "Plenjouse",
   "as_written": "Plenjouse",
   "role": "depicted",
   "country": null,
   "lat": null,
   "lon": null
  }
 ]
}


## 3. Pass 2 — evidence-based preprocessing audit
Apertus answers concrete yes/no checks with a quoted piece of evidence; deterministic detectors cover the mechanical checks (encoding damage, capitals, labelled sub-fields, packed fields, dotted dates). Final flag = Apertus OR rule.

In [4]:
araw, ameta = ap.chat_json(classify.build_audit_messages(rec))
checks, pre, personal = classify.audit_to_flags(araw, rec)
pd.DataFrame([{"preprocessing": k, "needed": v["needed"], "priority": v["priority"], "reason": v["reason"]} for k, v in pre.items()])

,preprocessing,needed,priority,reason
0,transcription_normalization,False,low,NaN
1,structure_extraction,False,low,NaN
2,schema_harmonization,False,low,NaN
3,linked_data_entity_linking,True,low,named organizations or places (apertus): hiera...
4,date_geolocation_parsing,True,medium,"nonstandard dates (apertus): expressed: ""1914...."
5,anonymization_verification,False,low,NaN


## 4. Derived facts, gazetteer and the story trail

In [5]:
derived = classify.derived_fields(rec)
cache = enrich.Cache(cfg["paths"]["data_dir"] / "enrich_cache.json")
places = [dict(p, geo=enrich.geocode(p["name"], p.get("country"), cache, cfg)) for p in result["places"]]
story = enrich.build_story({"apertus": result, "derived": derived}, rec, places)
print("Archive path:", " > ".join(derived["archival_location"]["path"]))
pd.DataFrame(story)[["event", "date", "place", "lat", "lon", "source"]]

Archive path: Archiv des Schweizerischen Bundesstaates ab 1848 > Landesverteidigung (1848-1950) > Stabsdienste, Dienstzweige, Hilfsdienste. Einsatz der Armee > Massnahmen bei aussenpolitischen Spannungen, Grenzbesetzungen, Aktivdienste > Aktivdienst 1914-1918 > Aktivdienstberichte, Aktivdiensterinnerungen > Aktivdiensterinnerungen, Fotosammlungen, Tagebücher > Fotosammlungen über den Aktivdienst > Bildarchiv der geografisch-archivistischen Sektion des Armeestabes [Aktivdienst und Kriegsgeschehen] > Negativplatten und digitale Abzüge 13 x 18 cm; Nr. 15 - 5758; Sch. 1 - 261


,event,date,place,lat,lon,source
0,creation,1914-01-01/1918-12-31,Plenjouse,NaN,NaN,apertus
1,accession,NaN,Bern,46.94198,7.44291,record
2,holding,NaN,"Archivstrasse 24, 3003 Bern",46.94198,7.44291,record


## 5. The 100-record AI-ready dataset

In [6]:
data = cfg["paths"]["data_dir"] / "dataset"
df = pd.read_csv(data / "cora_dataset.csv")
print(df.shape)
df[["id", "title_en", "document_type", "source_level", "date_created", "date_archived", "fonds"]].head(10)

(100, 38)


,id,title_en,document_type,source_level,date_created,date_archived,fonds
0,22406340,Loeche / High-voltage power line; Télé Journal...,TV news report,primary,1985-10-22,1998.0,Télévision Suisse Romande (1954-)
1,21730247,"Tagesschau - Téléjournal - Telegiornale (TS), ...",TV news report,primary,1965-11-23,1996.0,Schweizer Fernsehen DRS (1953-)
2,21690234,The Defeated,newsreel,primary,1974-07-05,1996.0,Stiftung Schweizer Filmwochenschau (1942-1975)
3,29668237,Judgment against Perincek,TV news report,primary,2007-03-09,2007.0,Schweizer Fernsehen DRS (1953-)
4,21810037,"Tagesschau (TS), 04.02.1985",TV news report,primary,1985-02-04,1996.0,Schweizer Fernsehen DRS (1953-)
5,21795934,Tagesschau - Téléjournal - Telegiornale (TS Ag...,TV news report,primary,1980-08-05,1996.0,Schweizer Fernsehen DRS (1953-)
6,21785789,"Tagesschau - Téléjournal - Telegiornale (TS), ...",TV news report,primary,1977-08-31,1996.0,Schweizer Fernsehen DRS (1953-)
7,21743659,"Tagesschau - Téléjournal - Telegiornale (TS), ...",TV news report,primary,1967-10-04,1996.0,Schweizer Fernsehen DRS (1953-)
8,29666896,Swissair Trial / Defense of Vreni Spoerry,TV news report,primary,2007-02-28,2007.0,Schweizer Fernsehen DRS (1953-)
9,21726160,"Tagesschau - Téléjournal - Telegiornale (TS), ...",TV news report,primary,1965-05-08,1996.0,Schweizer Fernsehen DRS (1953-)


In [7]:
pd.crosstab(df["stratum"], df["document_type"])

document_type,TV news report,administrative file,correspondence,minutes / protocol,newsreel,parliamentary record,photograph,radio broadcast,report
stratum,,,,,,,,,
audiovisual,35,0,0,0,4,0,0,1,0
other,0,1,1,1,3,23,0,0,1
still_image,0,0,0,0,1,0,29,0,0


In [8]:
df.filter(like="pre_").mean().mul(100).round(1).rename("% of records needing it").to_frame()

,% of records needing it
pre_transcription_normalization,42.0
pre_structure_extraction,43.0
pre_schema_harmonization,69.0
pre_linked_data_entity_linking,100.0
pre_date_geolocation_parsing,99.0
pre_anonymization_verification,12.0


## 6. Provenance: the edit log

In [9]:
log = pd.read_json(data / "edit_log.jsonl", lines=True)
print(log["actor"].value_counts().to_dict())
log[log["field"].isin(["title", "date_archived"])].head(8)

{'rule': 274, 'gazetteer': 82, 'apertus': 1}


,record_id,field,original,new,actor,reason,timestamp
1,22406340,date_archived,None,1998,rule,accession year taken from the accession number...,2026-10-18 21:51:00+00:00
5,21730247,date_archived,None,1996,rule,accession year taken from the accession number...,2026-10-18 21:51:00+00:00
8,21690234,date_archived,None,1996,rule,accession year taken from the accession number...,2026-10-18 21:51:00+00:00
11,29668237,date_archived,None,2007,rule,accession year taken from the accession number...,2026-10-18 21:51:00+00:00
14,21810037,date_archived,None,1996,rule,accession year taken from the accession number...,2026-10-18 21:51:00+00:00
18,21795934,date_archived,None,1996,rule,accession year taken from the accession number...,2026-10-18 21:51:00+00:00
21,21785789,date_archived,None,1996,rule,accession year taken from the accession number...,2026-10-18 21:51:00+00:00
25,21743659,date_archived,None,1996,rule,accession year taken from the accession number...,2026-10-18 21:51:00+00:00


## 7. Quality check summary

In [10]:
qc = json.loads((cfg["paths"]["data_dir"] / "qc_report.json").read_text(encoding="utf-8"))["summary"]
keys = ["processed_ok_pct", "structural_error_pct", "resume_max_3_sentences_pct", "entity_grounding_strict_pct",
        "entity_grounding_pct", "date_created_in_range_pct", "date_archived_correct_pct", "places_resolved_pct",
        "model_coord_error_km_median", "creator_agreement", "publisher_agreement"]
pd.Series({k: qc[k] for k in keys}).to_frame("value")

,value
processed_ok_pct,99.0
structural_error_pct,1.0
resume_max_3_sentences_pct,100.0
entity_grounding_strict_pct,92.0
entity_grounding_pct,100.0
date_created_in_range_pct,100.0
date_archived_correct_pct,24.0
places_resolved_pct,82.8
model_coord_error_km_median,1.8
creator_agreement,40/44


In [11]:
mr = qc["manual_review"]
pd.Series(mr["field_accuracy"]).rename("manual review: % correct").to_frame()

,manual review: % correct
creator_publisher,80.0
dates,96.0
document_type,100.0
entities,76.0
personal_data,88.0
places,91.3
resume,68.0
source_level,100.0
title,88.0
